# ⚡ Sales Prediction System: Preprocessing & ML Workflow

This notebook demonstrates the step-by-step pipeline for loading, cleaning, feature engineering, and model training in the **Sales Prediction System**. This modular workflow powers the interactive Streamlit dashboard backend.

## 🛠️ Step 1: Import Dependencies
First, we import the core scientific computing, machine learning, and natural language processing libraries.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from textblob import TextBlob
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import joblib

## 🧼 Step 2: Automatic Data Cleaning & Formatting
Here we define our text normalization rules (merging variants like 'headphones', 'HEADPHONES', 'Head phone' into a single standard class) and configure `SimpleImputer` to run median imputation on numeric features and mode imputation on categorical features. We also handle date segmentations and inject simulated discount percentages since the raw Flipkart dataset does not include discount metrics out-of-the-box.

In [ ]:
def clean_product_name(product):
    if not isinstance(product, str):
        return str(product)
    p_clean = product.strip().lower()
    if 'headphone' in p_clean or 'head phone' in p_clean:
        return 'headphones'
    if 'shoe' in p_clean:
        return 'shoes'
    if 'refrigerator' in p_clean or 'fridge' in p_clean:
        return 'refrigerator'
    return p_clean

raw_data_path = '../data/Flipkart_Sales_Uncleaned.csv'
df = pd.read_csv(raw_data_path)
df.columns = df.columns.str.strip()

# 1. Drop Duplicates
df = df.drop_duplicates()

# 2. Impute Numeric with Median & Text with Mode
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

if numeric_cols:
    df[numeric_cols] = SimpleImputer(strategy='median').fit_transform(df[numeric_cols])
if categorical_cols:
    df[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(df[categorical_cols])

# 3. Strip Whitespace & Normalize Products
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()
df['Product'] = df['Product'].apply(clean_product_name)

# 4. Generate Deterministic Simulated Discount Column
np.random.seed(42)
df['discount_pct'] = np.random.choice([0.0, 5.0, 10.0, 15.0, 20.0, 25.0], size=len(df))

# 5. Parse Chronology Features
df['Date'] = pd.to_datetime(df['Date'], errors='coerce').ffill().bfill()
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day
df['weekday'] = df['Date'].dt.weekday
df['weekend'] = df['weekday'].apply(lambda x: 1 if x in [5, 6] else 0)
df['quarter'] = df['Date'].dt.quarter

## 📐 Step 3: Feature Engineering
We calculate dynamic metric indicators: `sales_per_quantity`, segment unit pricing categories (Low, Medium, High), assign meteorological quarters (`season`), and compute transactional consumer sentiment polarity using `TextBlob`.

In [ ]:
df['sales_per_quantity'] = df['Sales_Value_INR'] / df['Quantity'].replace(0, 1)

def categorize_price(price):
    if price <= 500: return 'Low'
    elif price <= 5000: return 'Medium'
    else: return 'High'

df['price_category'] = df['Unit_Price_INR'].apply(categorize_price)
df['season'] = df['quarter'].apply(lambda q: f'Q{q}')

def calculate_sentiment(row):
    prod = str(row.get('Product', ''))
    qty = float(row.get('Quantity', 5))
    disc = float(row.get('discount_pct', 0))
    if qty > 10 and disc > 15:
        text = f"Excellent discount on {prod}! Fast delivery and highly satisfied with this purchase."
    elif qty < 3 and disc == 0:
        text = f"Decent {prod}, but price was somewhat high and no discount was offered."
    else:
        text = f"Good quality {prod}. Functions perfectly and meets all expectations."
    return TextBlob(text).sentiment.polarity

df['sentiment_score'] = df.apply(calculate_sentiment, axis=1)
print("Cleaned Data Columns:", df.columns.tolist())
df.head(3)

## 🧠 Step 4: Machine Learning Model Training & Evaluation
We select the target and predictors, encode our categorical columns using `LabelEncoder`, split the data into training/validation sections, and fit **Linear Regression**, **Decision Tree**, and **Random Forest** models to identify the highest $R^2$ performer.

In [ ]:
features = ['Quantity', 'Unit_Price_INR', 'discount_pct', 'State', 'City', 'Product', 'year', 'month', 'day', 'weekend']
target = 'Sales_Value_INR'

X = df[features].copy()
y = df[target].copy()

# Categorical Label Encodings
encoders = {}
for col in ['State', 'City', 'Product']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=12, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
}

performance = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    performance[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    
    print(f"[{name}] R2: {r2:.4f} | MAE: {mae:,.2f} | RMSE: {rmse:,.2f}")

## 💾 Step 5: Serialize Best Model
Finally, we extract the best model, bundle it with our categorical encoders, and save it for deployment in our interactive Streamlit application.

In [ ]:
best_model_name = max(performance, key=lambda k: performance[k]['R2'])
best_model = models[best_model_name]
print(f"Selected: {best_model_name} (R2={performance[best_model_name]['R2']:.4f})")

model_state = {
    'model': best_model,
    'model_name': best_model_name,
    'encoders': encoders,
    'features': features,
    'performance': performance
 }

os.makedirs('../models', exist_ok=True)
joblib.dump(model_state, '../models/sales_model.pkl')
print("Model saved successfully to '../models/sales_model.pkl'!")